# Fall detection for care-home night alerts: training pipeline

A camera in a resident's room takes an image and the model decides whether to send the night
nurse. **Falling is the positive class.** Section numbers match REPORT.md, where the reasoning
behind every choice is written up.

Run end to end with `uv run python main.py train`, or on Colab with a T4 GPU.

## Setup

On Colab: Runtime > Change runtime type > T4, and add a secret called `ROBOFLOW_API_KEY` with
notebook access on. Locally the key comes from `.env`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab starts in an empty folder, so fetch the repo to get api_key.py and the weights/ folder.
    if not Path("api_key.py").exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Allan-Data-science/daiist-assignment-2.git"],
            check=True,
        )
        os.chdir("daiist-assignment-2")
    # Plain pip, not %pip: main.py runs this notebook as an ordinary script, where magics do not
    # exist. No uv sync either, because Colab brings its own GPU build of PyTorch. Ultralytics is
    # pinned to the same range as pyproject.toml so the weights load in the local app.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4,<8.5", "roboflow"],
        check=True,
    )
    # api_key.py checks the environment variable first, so the Colab secret goes there.
    from google.colab import userdata

    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

from api_key import load_api_key

API_KEY = load_api_key()
# Print the shape of the key, never the key.
print(f"key loaded: {len(API_KEY)} characters")

## 2. Dataset

| | training data | external test (test split only) |
|---|---|---|
| workspace | `falldetection-bwfdz` | `ynt423` |
| project | `fall-detection-ehgga` | `labels-6tnj1` |
| version | `1` | `1` |
| classes | `Falling`, `Sitting`, `Standing` | `Fall`, `Sitting`, `Standing` |
| licence | CC BY 4.0 | CC BY 4.0 |

I use the external set only for testing, never for training. Section 3.2 shows why I need it.

In [ ]:
import yaml
from roboflow import Roboflow


def download(workspace, project, version):
    dataset_dir = Path(f"datasets/{project}-{version}").resolve()
    if not dataset_dir.exists():
        rf = Roboflow(api_key=API_KEY)
        rf.workspace(workspace).project(project).version(version).download(
            "yolov11", location=str(dataset_dir)
        )
    # Roboflow writes split paths like "../train/images", which point outside the dataset
    # folder. Rewrite them relative to an explicit root so Ultralytics finds the images.
    data_yaml = dataset_dir / "data.yaml"
    config = yaml.safe_load(data_yaml.read_text())
    config.update(path=str(dataset_dir), train="train/images", val="valid/images",
                  test="test/images")
    data_yaml.write_text(yaml.safe_dump(config))
    return dataset_dir, data_yaml, config["names"]


DATASET_DIR, DATA_YAML, CLASS_NAMES = download("falldetection-bwfdz", "fall-detection-ehgga", 1)
EXTERNAL_DIR, EXTERNAL_YAML, EXTERNAL_NAMES = download("ynt423", "labels-6tnj1", 1)

POSITIVE = "Falling"
# Both datasets index Fall(ing), Sitting, Standing as 0, 1, 2, so our model's class ids apply to
# the external labels unchanged. Stop here if that ever stops being true.
assert [n[:4].lower() for n in EXTERNAL_NAMES] == [n[:4].lower() for n in CLASS_NAMES]
print(CLASS_NAMES, EXTERNAL_NAMES)

## 3. What the data tells us

### 3.1 Describe it before training

Instances per class, images per split and box size. Each training photo comes with two augmented
copies, so I also count the original photos (the part of the file name before `.rf.`).

In [ ]:
from collections import Counter


def describe_split(dataset_dir, split):
    label_files = sorted((dataset_dir / split / "labels").glob("*.txt"))
    instances, box_areas, empty, crowded = Counter(), [], 0, 0
    for f in label_files:
        # A YOLO label file is one box per line: "class_id x y w h", all relative to the image.
        lines = [line.split() for line in f.read_text().splitlines() if line.strip()]
        if not lines:
            empty += 1
        if len(lines) > 1:
            crowded += 1
        for parts in lines:
            instances[CLASS_NAMES[int(parts[0])]] += 1
            coords = [float(x) for x in parts[1:]]
            if len(coords) == 4:
                w, h = coords[2], coords[3]
            else:
                # Some external labels are polygons ("cls x1 y1 x2 y2 ..."): use their bounding box.
                w, h = max(coords[0::2]) - min(coords[0::2]), max(coords[1::2]) - min(coords[1::2])
            box_areas.append(w * h)
    originals = {f.name.split(".rf.")[0] for f in label_files}
    box_areas.sort()
    return {
        "images": len(label_files),
        "originals": len(originals),
        "empty": empty,
        "multi_person": crowded,
        **{name: instances[name] for name in CLASS_NAMES},
        "median_box": box_areas[len(box_areas) // 2],
    }


dataset_stats = {split: describe_split(DATASET_DIR, split) for split in ("train", "valid", "test")}
dataset_stats["external"] = describe_split(EXTERNAL_DIR, "test")

print(f"{'split':<10}{'images':>8}{'originals':>11}{'empty':>7}{'2+ people':>11}", end="")
print("".join(f"{name:>10}" for name in CLASS_NAMES) + f"{'median box':>12}")
for split, s in dataset_stats.items():
    print(f"{split:<10}{s['images']:>8}{s['originals']:>11}{s['empty']:>7}{s['multi_person']:>11}",
          end="")
    print("".join(f"{s[name]:>10}" for name in CLASS_NAMES) + f"{s['median_box']:>12.0%}")

### 3.2 Are the test images really new?

File names miss burst photos, so I compare images instead. Each image becomes a 256-bit
difference hash, and I measure how far each test image is from its closest training image. A few
bits out of 256 means it is practically the same picture.

In [ ]:
import numpy as np
from PIL import Image


def image_hash(path):
    grey = np.asarray(Image.open(path).convert("L").resize((17, 16)), dtype=np.int16)
    return (grey[:, 1:] > grey[:, :-1]).flatten()


train_hashes = np.array([image_hash(p) for p in (DATASET_DIR / "train" / "images").glob("*")])


def nearest_train_distance(image_dir):
    return np.array([(train_hashes != image_hash(p)).sum(1).min() for p in image_dir.glob("*")])


near_duplicates = {}
print(f"{'split':<10}{'images':>8}{'within 10 bits':>16}{'median distance':>17}")
for split, image_dir in [("valid", DATASET_DIR / "valid" / "images"),
                         ("test", DATASET_DIR / "test" / "images"),
                         ("external", EXTERNAL_DIR / "test" / "images")]:
    d = nearest_train_distance(image_dir)
    near_duplicates[split] = {"images": len(d), "within_10": int((d <= 10).sum()),
                              "median": int(np.median(d))}
    print(f"{split:<10}{len(d):>8}{int((d <= 10).sum()):>16}{int(np.median(d)):>17}")

## 4. Finetuning

I start from COCO-pretrained `yolo11n`, so the model already knows what a person looks like and
only has to learn posture. Patience 10 stops training if validation stops improving.
`EPOCHS=1` in the environment gives a quick smoke test.

In [ ]:
import shutil

from ultralytics import YOLO

EPOCHS = int(os.environ.get("EPOCHS", 50))
IMGSZ, BATCH, PATIENCE, SEED = 640, 16, 10, 0

model = YOLO("yolo11n.pt")
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    project=str(Path("runs").resolve()),
    name="fall",
    exist_ok=True,
)

# runs/ is not committed. The app and the grader load weights/best.pt, so copy it there.
best_run_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
WEIGHTS = Path("weights/best.pt")
shutil.copy(best_run_weights, WEIGHTS)
print(f"saved {WEIGHTS} ({WEIGHTS.stat().st_size / 1e6:.1f} MB)")

## 5. Per-class results

Falling is the class that matters. The internal test split is leaked (section 3.2), so treat it
as an upper bound. The external set is the realistic number.

In [ ]:
import json

finetuned = YOLO(str(WEIGHTS))


def class_metrics(data_yaml, name):
    # Box-level precision, recall and AP per class, plus the confusion matrix, on one test split.
    result = finetuned.val(data=str(data_yaml), split="test", imgsz=IMGSZ, plots=True, name=name)
    box = result.box
    per_class = {
        cls: {"precision": float(box.p[i]), "recall": float(box.r[i]),
              "ap50": float(box.ap50[i]), "ap50_95": float(box.maps[i])}
        for i, cls in enumerate(CLASS_NAMES)
    }
    print(f"{name}: overall  mAP50 {box.map50:.3f}   mAP50-95 {box.map:.3f}")
    print(f"{'class':<10}{'precision':>11}{'recall':>9}{'AP50':>8}{'AP50-95':>10}")
    for cls, m in per_class.items():
        print(f"{cls:<10}{m['precision']:>11.3f}{m['recall']:>9.3f}"
              f"{m['ap50']:>8.3f}{m['ap50_95']:>10.3f}")
    print(f"confusion matrix plot: {result.save_dir}/confusion_matrix.png\n")
    return {
        "overall": {"map50": float(box.map50), "map50_95": float(box.map)},
        "per_class": per_class,
        # Rows are predicted class, columns are true class; the last row/column is background.
        "confusion_matrix": result.confusion_matrix.matrix.astype(int).tolist(),
    }


test_results = {"internal": class_metrics(DATA_YAML, "internal"),
                "external": class_metrics(EXTERNAL_YAML, "external")}

# Everything the app needs later goes in outputs/, which is committed alongside the weights.
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
(OUTPUTS / "test_metrics.json").write_text(
    json.dumps(
        {
            "dataset": dataset_stats,
            "near_duplicates": near_duplicates,
            **test_results,
            "matrix_labels": CLASS_NAMES + ["background"],
            "train_settings": {"epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
                               "patience": PATIENCE, "seed": SEED},
        },
        indent=2,
    )
)

### 4.1 Baselines: one fall score per image

(This sits after section 5 because it reuses the finetuned model loaded there.)

The nurse acts on one image at a time, so each system gives one fall score per image. An image
counts as a fall if anyone in it has fallen. An alert is raised when the score is at or above the
cut.

| system | fall score |
|---|---|
| finetuned | highest `Falling` confidence, 0 if none |
| COCO any person | highest `person` confidence, 0 if none |
| COCO wide box | highest `person` box that is wider than tall, 0 if none |
| predict nothing | always 0 |

I keep predictions down to 1% confidence so the sweep can see the borderline cases.

In [ ]:
coco = YOLO("yolo11n.pt")  # the same checkpoint we finetuned from, used as it is
COCO_PERSON = 0  # "person" is class 0 in COCO
FALLING = CLASS_NAMES.index(POSITIVE)
SYSTEMS = ["finetuned", "coco_person", "coco_wide", "nothing"]


def max_conf(boxes):
    return max((float(b.conf) for b in boxes), default=0.0)


def is_fall(label_path):
    # Ground truth for one image: does any label in it say Fall(ing)?
    ids = {int(line.split()[0]) for line in label_path.read_text().splitlines() if line.strip()}
    return FALLING in ids


def score_test_set(dataset_dir):
    cases = []
    for path in sorted((dataset_dir / "test" / "images").glob("*")):
        case = {"image": path.name,
                "truth": is_fall(dataset_dir / "test" / "labels" / f"{path.stem}.txt")}

        ours = finetuned.predict(path, conf=0.01, imgsz=IMGSZ, verbose=False)[0].boxes
        case["finetuned"] = max_conf([b for b in ours if int(b.cls) == FALLING])

        people = [b for b in coco.predict(path, conf=0.01, verbose=False)[0].boxes
                  if int(b.cls) == COCO_PERSON]
        case["coco_person"] = max_conf(people)
        # xywh is centre x, centre y, width, height. Someone lying on the floor is wider than tall.
        case["coco_wide"] = max_conf([b for b in people if b.xywh[0][2] > b.xywh[0][3]])

        case["nothing"] = 0.0
        cases.append(case)
    print(f"{dataset_dir.name}: {len(cases)} images, {sum(c['truth'] for c in cases)} with a fall")
    return cases


test_sets = {"internal": score_test_set(DATASET_DIR), "external": score_test_set(EXTERNAL_DIR)}

### 4.2 The four systems at two example cuts

The cut I actually use is chosen by cost in section 6.

In [ ]:
def confusion(cases, system, cut):
    # Positive class is a fall. A score at or above the cut raises an alert.
    tp = sum(1 for c in cases if c[system] >= cut and c["truth"])
    fp = sum(1 for c in cases if c[system] >= cut and not c["truth"])
    fn = sum(1 for c in cases if c[system] < cut and c["truth"])
    tn = sum(1 for c in cases if c[system] < cut and not c["truth"])
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "recall": tp / (tp + fn),
        "specificity": tn / (tn + fp),
        "precision": tp / (tp + fp) if tp + fp else float("nan"),
        "accuracy": (tp + tn) / len(cases),
    }


for name, cases in test_sets.items():
    for cut in (0.25, 0.5):
        print(f"{name} test set, cut {cut}")
        print(f"{'system':<13}{'TP':>5}{'FP':>5}{'FN':>5}{'TN':>5}"
              f"{'recall':>9}{'specificity':>13}{'precision':>11}{'accuracy':>10}")
        for system in SYSTEMS:
            s = confusion(cases, system, cut)
            precision = "-" if s["precision"] != s["precision"] else f"{s['precision']:.1%}"
            print(f"{system:<13}{s['tp']:>5}{s['fp']:>5}{s['fn']:>5}{s['tn']:>5}"
                  f"{s['recall']:>9.1%}{s['specificity']:>13.1%}{precision:>11}"
                  f"{s['accuracy']:>10.1%}")
        print()

# Per-image scores are what the cost sweep and the dashboard are built from.
(OUTPUTS / "test_scores.json").write_text(json.dumps(test_sets, indent=2))

## 6. Cost matrix and operating point

| outcome | cost | what happens |
|---|---|---|
| TP | EUR 5 | nurse goes and finds the fall |
| FP | EUR 5 | nurse goes for nothing |
| FN | EUR 2,000 | resident stays on the floor until the next round |
| TN | EUR 0 | no fall, no trip |

I assume a 1% base rate. The test sets have 33% and 18% falls, so I take recall and specificity
from them (these do not depend on the base rate) and project them onto 1%:

```
TP = p * recall                   FN = p * (1 - recall)
FP = (1 - p) * (1 - specificity)  TN = (1 - p) * specificity
```

To avoid picking and reporting on the same images, I split the external set in two (stratified,
seed 0). The cut is chosen on the tune half and the cost is reported on the report half.

In [ ]:
import random

COST_TP, COST_FP, COST_FN, COST_TN = 5.0, 5.0, 2000.0, 0.0
BASE_RATE = 0.01

# Cuts from 0.01 to 1.01 (never alert) in steps of 0.01. A cut of 0 is left out: a score of 0
# means the model found no box at all, and alerting on that is "check every image", which is
# priced separately as the everyone-reviews design, not as a model threshold.
CUTS = [round(i / 100, 2) for i in range(1, 102)]

# Stratified 50/50 split of the external set: shuffle falls and non-falls separately with a fixed
# seed, so both halves keep the 18% fall share and the split is the same on every run.
rng = random.Random(SEED)
external_falls = [c for c in test_sets["external"] if c["truth"]]
external_others = [c for c in test_sets["external"] if not c["truth"]]
rng.shuffle(external_falls)
rng.shuffle(external_others)
TUNE = external_falls[: len(external_falls) // 2] + external_others[: len(external_others) // 2]
REPORT = external_falls[len(external_falls) // 2 :] + external_others[len(external_others) // 2 :]
print(f"tune half: {len(TUNE)} images, {sum(c['truth'] for c in TUNE)} falls")
print(f"report half: {len(REPORT)} images, {sum(c['truth'] for c in REPORT)} falls")


def cost_per_1000(stats, base_rate=BASE_RATE, cost_fn=COST_FN):
    falls, others = 1000 * base_rate, 1000 * (1 - base_rate)
    tp, fn = falls * stats["recall"], falls * (1 - stats["recall"])
    fp, tn = others * (1 - stats["specificity"]), others * stats["specificity"]
    return tp * COST_TP + fp * COST_FP + fn * cost_fn + tn * COST_TN


def sweep(cases, system, base_rate=BASE_RATE, cost_fn=COST_FN):
    # One row per cut: the cut, its expected cost per 1,000 cases, and the confusion on these cases.
    rows = []
    for cut in CUTS:
        s = confusion(cases, system, cut)
        rows.append({"cut": cut, "cost": cost_per_1000(s, base_rate, cost_fn), **s})
    return rows


def cheapest(rows):
    return min(rows, key=lambda r: r["cost"])


def tuned_then_reported(system, base_rate=BASE_RATE, cost_fn=COST_FN):
    # Pick the cut on the tune half, then measure that same cut on the report half.
    cut = cheapest(sweep(TUNE, system, base_rate, cost_fn))["cut"]
    s = confusion(REPORT, system, cut)
    return {"cut": cut, "cost": cost_per_1000(s, base_rate, cost_fn), **s}


sweeps = {
    name: {system: sweep(cases, system) for system in SYSTEMS}
    for name, cases in [("internal", test_sets["internal"]), ("tune", TUNE), ("report", REPORT)]
}
reported = {system: tuned_then_reported(system) for system in SYSTEMS}
CHOSEN = reported["finetuned"]

print(f"\neach system: cut chosen on the tune half, cost measured on the report half, "
      f"base rate {BASE_RATE:.0%}")
print(f"{'system':<13}{'cut':>6}{'tune EUR':>10}{'recall':>9}{'specificity':>13}{'report EUR':>12}")
for system in SYSTEMS:
    r = reported[system]
    tune_cost = cheapest(sweeps["tune"][system])["cost"]
    print(f"{system:<13}{r['cut']:>6}{tune_cost:>10,.0f}{r['recall']:>9.1%}"
          f"{r['specificity']:>13.1%}{r['cost']:>12,.0f}")

# A person looking at every image catches every fall and pays EUR 5 per image.
CHECK_ALL = 1000 * BASE_RATE * COST_TP + 1000 * (1 - BASE_RATE) * COST_FP
print(f"{'check all':<13}{'-':>6}{CHECK_ALL:>10,.0f}{1:>9.1%}{0:>13.1%}{CHECK_ALL:>12,.0f}")

internal_best = cheapest(sweeps["internal"]["finetuned"])
print(f"\nfor reference, leaked internal split: finetuned cheapest cut {internal_best['cut']} "
      f"at EUR {internal_best['cost']:,.0f} per 1,000")

In [ ]:
import matplotlib

matplotlib.use("Agg")  # no window: main.py runs this as a plain script
import matplotlib.pyplot as plt

panels = [("internal", "internal test (leaked)"), ("tune", "external, tune half"),
          ("report", "external, report half")]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, (name, title) in zip(axes, panels):
    for system in SYSTEMS:
        rows = sweeps[name][system]
        ax.plot([r["cut"] for r in rows], [r["cost"] for r in rows], label=system)
    ax.axhline(CHECK_ALL, color="grey", linestyle="--", label="person checks every image")
    if name == "internal":
        ax.scatter([internal_best["cut"]], [internal_best["cost"]], color="black", zorder=3,
                   label=f"cheapest cut {internal_best['cut']}")
    else:
        cost_at_chosen = next(r["cost"] for r in sweeps[name]["finetuned"]
                              if r["cut"] == CHOSEN["cut"])
        ax.scatter([CHOSEN["cut"]], [cost_at_chosen], color="black", zorder=3,
                   label=f"chosen cut {CHOSEN['cut']} (from tune half)")
    ax.set_title(title)
    ax.set_xlabel("confidence threshold (alert at or above)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("expected cost, EUR per 1,000 cases")
fig.suptitle(f"Cost against threshold at a {BASE_RATE:.0%} base rate")
fig.tight_layout()
fig.savefig(OUTPUTS / "cost_curve.png", dpi=120)

### 6.1 What would change the answer

I re-run the sweep for other FN costs and base rates, re-choosing the cut on the tune half each
time, to see where the cheapest option changes.

In [ ]:
def print_sensitivity(label, values, fmt):
    print(f"{label:>12}{'cut':>6}{'recall':>9}{'finetuned':>11}{'coco_person':>13}"
          f"{'coco_wide':>11}{'check all':>11}{'cheapest':>13}")
    rows = []
    for value in values:
        costs = {s: tuned_then_reported(s, **value) for s in SYSTEMS}
        base_rate = value.get("base_rate", BASE_RATE)
        check_all = 1000 * base_rate * COST_TP + 1000 * (1 - base_rate) * COST_FP
        options = {**{s: costs[s]["cost"] for s in SYSTEMS}, "check all": check_all}
        winner = min(options, key=options.get)
        ours = costs["finetuned"]
        print(f"{fmt(value):>12}{ours['cut']:>6}{ours['recall']:>9.1%}{ours['cost']:>11,.0f}"
              f"{costs['coco_person']['cost']:>13,.0f}{costs['coco_wide']['cost']:>11,.0f}"
              f"{check_all:>11,.0f}{winner:>13}")
        rows.append({**value, **options, "finetuned_cut": ours["cut"], "winner": winner})
    print()
    return rows


FN_COSTS = [250, 500, 1000, 2000, 5000, 10000]
BASE_RATES = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.05]

print(f"FN cost varied, base rate fixed at {BASE_RATE:.0%} (EUR per 1,000, report half)")
fn_sensitivity = print_sensitivity("FN cost", [{"cost_fn": c} for c in FN_COSTS],
                                   lambda v: f"EUR {v['cost_fn']:,}")
print(f"base rate varied, FN cost fixed at EUR {COST_FN:,.0f} (EUR per 1,000, report half)")
rate_sensitivity = print_sensitivity("base rate", [{"base_rate": r} for r in BASE_RATES],
                                     lambda v: f"{v['base_rate']:.2%}")

In [ ]:
(OUTPUTS / "cost_sweep.json").write_text(
    json.dumps(
        {
            "costs": {"tp": COST_TP, "fp": COST_FP, "fn": COST_FN, "tn": COST_TN},
            "base_rate": BASE_RATE,
            "check_all": CHECK_ALL,
            "split": {"seed": SEED, "tune": [c["image"] for c in TUNE],
                      "report": [c["image"] for c in REPORT]},
            "sweeps": sweeps,
            "reported": reported,
            "chosen": CHOSEN,
            "internal_cheapest": internal_best,
            "fn_sensitivity": fn_sensitivity,
            "rate_sensitivity": rate_sensitivity,
        },
        indent=2,
    )
)

## 7. What a person still has to check

Each image goes one of three ways:

| outcome | when | what happens |
|---|---|---|
| pass | below the review line | nothing |
| review | between the lines | staff check the image on a screen (about 60 s) |
| nurse | at or above the nurse line | the nurse goes straight to the room |

Assumptions: a review costs EUR 0.40, the reviewer catches 95% of falls and clears 99% of
non-falls (class notebook figures, not measured), and at most 75% of images go to review.

The review line is 0, because about 27% of external falls get no `Falling` box at all, so any line
above 0 lets real falls through unseen. The nurse line is the cheapest one on the tune half within
the 75% cap. The cap is there because the person watching the screen also does rounds, and it
sends the clearest cases straight to the nurse.

In [ ]:
REVIEW_COST, REVIEWER_RECALL, REVIEWER_SPECIFICITY = 0.40, 0.95, 0.99
REVIEW_CAP = 0.75

# A 100-bed home whose motion trigger fires about 150 times per bed per year (Section 5 base rate).
IMAGES_PER_NIGHT = 100 * 150 / 365
MINUTES_PER_REVIEW, MINUTES_PER_TRIP = 1, 10


def three_way(cases, low, high, system="finetuned", review_cost=REVIEW_COST,
              reviewer_recall=REVIEWER_RECALL, base_rate=BASE_RATE, cost_fn=COST_FN):
    # Per 1,000 cases at the real base rate. Zone shares are measured separately for falls and
    # non-falls, so like recall and specificity they survive the move to the real base rate.
    def zones(scores):
        n = len(scores)
        return (sum(x < low for x in scores) / n,
                sum(low <= x < high for x in scores) / n,
                sum(x >= high for x in scores) / n)

    fall_pass, fall_review, fall_fail = zones([c[system] for c in cases if c["truth"]])
    other_pass, other_review, other_fail = zones([c[system] for c in cases if not c["truth"]])
    falls, others = 1000 * base_rate, 1000 * (1 - base_rate)

    # A fall is missed if it is passed, or reviewed and the reviewer does not spot it.
    fn = falls * (fall_pass + fall_review * (1 - reviewer_recall))
    # The nurse goes for every fail, and for every review the reviewer flags.
    trips = (falls * (fall_fail + fall_review * reviewer_recall)
             + others * (other_fail + other_review * (1 - REVIEWER_SPECIFICITY)))
    reviews = falls * fall_review + others * other_review
    return {
        "low": low,
        "high": high,
        "cost": fn * cost_fn + trips * COST_TP + reviews * review_cost,
        "falls_caught": 1 - fn / falls,
        "review_share": reviews / 1000,
        "trip_share": trips / 1000,
    }


def best_fail_line(cap, **assumptions):
    # Cheapest fail line on the tune half within the review cap, then measured on the report half.
    allowed = [three_way(TUNE, 0, high, **assumptions) for high in CUTS]
    allowed = [d for d in allowed if d["review_share"] <= cap]
    high = min(allowed, key=lambda d: d["cost"])["high"]
    return three_way(REPORT, 0, high, **assumptions)


DESIGN = best_fail_line(REVIEW_CAP)
print(f"chosen design: review below {DESIGN['high']}, nurse at or above it")

### 7.1 The price of the cap

The same sweep for other caps.

In [ ]:
print(f"{'cap':>5}{'fail line':>11}{'EUR/1,000':>11}{'reviewed':>10}{'falls caught':>14}"
      f"{'reviews/night':>15}{'nurse trips/night':>19}")
cap_table = []
for cap in (1.0, 0.9, 0.8, 0.75, 0.7, 0.6, 0.5):
    d = best_fail_line(cap)
    cap_table.append({"cap": cap, **d})
    marker = "  <- chosen" if cap == REVIEW_CAP else ""
    print(f"{cap:>5.0%}{d['high']:>11}{d['cost']:>11,.0f}{d['review_share']:>10.0%}"
          f"{d['falls_caught']:>14.0%}{d['review_share'] * IMAGES_PER_NIGHT:>15.0f}"
          f"{d['trip_share'] * IMAGES_PER_NIGHT:>19.1f}{marker}")

### 7.2 Against the alternatives

Doing nothing, the model alone, the nurse walking to every image, staff reviewing every image, and
my design. All on the report half.

In [ ]:
alternatives = {
    # No reviews and no trips: every fall is missed.
    "do nothing": {"cost": 1000 * BASE_RATE * COST_FN, "falls_caught": 0.0,
                   "review_share": 0.0, "trip_share": 0.0},
    # Equal lines mean no review zone: below the cut is passed, at or above it the nurse goes.
    "model alone": three_way(REPORT, CHOSEN["cut"], CHOSEN["cut"]),
    "nurse walks to every image": three_way(REPORT, 0, 0),
    "staff review every image": three_way(REPORT, 0, 1.01),
    f"chosen design (cap {REVIEW_CAP:.0%})": DESIGN,
}

print(f"{'design':<30}{'EUR/1,000':>11}{'falls caught':>14}{'reviews/night':>15}"
      f"{'review min':>12}{'trips/night':>13}{'trip min':>10}")
for name, d in alternatives.items():
    reviews = d["review_share"] * IMAGES_PER_NIGHT
    trips = d["trip_share"] * IMAGES_PER_NIGHT
    print(f"{name:<30}{d['cost']:>11,.0f}{d['falls_caught']:>14.0%}{reviews:>15.0f}"
          f"{reviews * MINUTES_PER_REVIEW:>12.0f}{trips:>13.1f}{trips * MINUTES_PER_TRIP:>10.0f}")

### 7.3 Reviewer sensitivity

Reviewer accuracy and review cost are assumptions, so I vary each with the cap fixed at 75%.

In [ ]:
reviewer_sensitivity = []
print(f"{'reviewer recall':>16}{'fail line':>11}{'EUR/1,000':>11}{'falls caught':>14}")
for recall in (0.99, 0.95, 0.90, 0.80, 0.70):
    d = best_fail_line(REVIEW_CAP, reviewer_recall=recall)
    reviewer_sensitivity.append({"reviewer_recall": recall, **d})
    print(f"{recall:>16.0%}{d['high']:>11}{d['cost']:>11,.0f}{d['falls_caught']:>14.0%}")

print(f"\n{'review cost':>16}{'fail line':>11}{'EUR/1,000':>11}{'falls caught':>14}")
for review_cost in (0.20, 0.40, 0.60, 1.00, 2.00):
    d = best_fail_line(REVIEW_CAP, review_cost=review_cost)
    reviewer_sensitivity.append({"review_cost": review_cost, **d})
    print(f"{'EUR %.2f' % review_cost:>16}{d['high']:>11}{d['cost']:>11,.0f}{d['falls_caught']:>14.0%}")

In [ ]:
(OUTPUTS / "review_design.json").write_text(
    json.dumps(
        {
            "assumptions": {"review_cost": REVIEW_COST, "reviewer_recall": REVIEWER_RECALL,
                            "reviewer_specificity": REVIEWER_SPECIFICITY, "review_cap": REVIEW_CAP,
                            "images_per_night": IMAGES_PER_NIGHT,
                            "minutes_per_review": MINUTES_PER_REVIEW,
                            "minutes_per_trip": MINUTES_PER_TRIP},
            "design": DESIGN,
            "cap_table": cap_table,
            "alternatives": alternatives,
            "reviewer_sensitivity": reviewer_sensitivity,
        },
        indent=2,
    )
)